# Comparing Summarization Models: Baseline vs. Fine-tuned

This notebook evaluates and compares the performance of:  
1. Pre-trained HuggingFace models (zero-shot and few-shot settings)  
2. Our fine-tuned model

We'll assess their performance using both standard metrics and more nuanced evaluations focused on factual accuracy and hallucination detection.

## Setup and Configuration

In [27]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.notebook import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from datasets import load_metric, Dataset
import nltk
from nltk.tokenize import sent_tokenize

# Add parent directory to path for importing project modules
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'scripts'))
from vector_db_manager import VectorDatabaseManager

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Configure paths
DB_PATH = '../data/chroma_db'
COLLECTION_NAME = 'financial_articles'
OUTPUT_DIR = '../models/bart_finetuned_opt'

# Base model to use for zero-shot and few-shot
BASE_MODEL = 'facebook/bart-large-cnn'

## Load Models and Test Dataset

We'll load both the baseline model from HuggingFace and our fine-tuned model, along with a test dataset for evaluation.

In [28]:
# Load base model and tokenizer from HuggingFace
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# Load fine-tuned model and tokenizer
try:
    finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR,early_stopping=True)
    print(f"Successfully loaded fine-tuned model from {OUTPUT_DIR}")
except Exception as e:
    print(f"Error loading fine-tuned model: {e}")
    print("Proceeding with only the base model for comparison.")
    finetuned_model = None
    finetuned_tokenizer = None

Successfully loaded fine-tuned model from ../models/bart_finetuned_opt


In [33]:
# Load test dataset from vector database
def load_test_data(db_path, collection_name, limit=20):
    """Load test data from the vector database."""
    # Create a VectorDatabaseManager instance
    db_manager = VectorDatabaseManager(
        db_path=db_path,
        collection_name=collection_name
    )
    
    # Use the query_database method to retrieve articles
    # Since you want all articles with summaries, use an empty query
    results = db_manager.query_database(
        query="",  # Empty query to match all documents
        n_results=limit,
        metadata_filter=None,  # No filtering by source
        include_summary=True
    )
    
    test_data = []
    # Process the results based on ChromaDB format
    if results and 'documents' in results and results['documents']:
        # Check if documents is a list of lists (typical ChromaDB format)
        documents = results['documents'][0] if isinstance(results['documents'][0], list) else results['documents']
        metadatas = results['metadatas'][0] if isinstance(results['metadatas'][0], list) else results['metadatas']
        
        for i, doc in enumerate(documents):
            metadata = metadatas[i]
            # Only include articles that have summaries
            if 'summary' in metadata and metadata['summary']:
                test_data.append({
                    'id': i,
                    'text': doc,
                    'summary': metadata['summary']
                })
    
    return test_data

# Load test data
test_data = load_test_data(DB_PATH, COLLECTION_NAME)
print(f"Loaded {len(test_data)} test examples")

# Display a sample
if test_data:
    example = test_data[2]
    print("\nSample article:\n", example['text'][:300], "...\n")
    print("Reference summary:\n", example['summary'])



Initializing ChromaDB with persistence directory: ../data/chroma_db
Loaded existing collection 'financial_articles'
Collection 'financial_articles' contains 3118 documents
Found 20 results. Showing top 3:

Result #1 (Similarity: -0.5309)
Title: How Governments Shape Markets
Source: Fool
Date: 2025-05-16 15:21:00
Chunk: 8 of 93
Link: https://www.fool.com/investing/2025/05/16/how-governments-shape-markets/
Creator: newsfeedback@fool.com (Motley Fool Staff)
Summary: Ricky Mulvey: I'm glad I'm glad to have that experience.
One of the themes that struck me from this is that, you know, people like to think, I'm a free markets person, or I want government involved with things.
I look mid century treasury markets and a pretty conservative head of the Federal Reserve who use policy to make those markets orderly.
Chris Hughes: I'm still writing a dissertation to finish a PhD it's on Fed history.
I'm going to sell my shares in Meta, and I'm going to give you that money for your national investmen

## Implement Summarization Strategies

We'll implement three summarization approaches:
1. Zero-shot: Using the model without special prompting
2. Few-shot: Providing examples before asking for a summary
3. Fine-tuned: Using our domain-specific trained model

In [34]:
def zero_shot_summarize(model, tokenizer, text, max_length=150, min_length=40):
    """Summarize text using a model without examples (zero-shot)."""
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    
    # Generate summary
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    
    # Decode the summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    return summary

def few_shot_summarize(model, tokenizer, text, examples, max_length=150, min_length=40):
    """Summarize text using a few examples in the prompt."""
    # Construct prompt with examples
    prompt = ""
    for example in examples:
        prompt += f"Article: {example['text']}\nSummary: {example['summary']}\n\n"
    
    # Add the new article to summarize
    prompt += f"Article: {text}\nSummary:"
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    # Extract just the summary part
    if "Summary:" in summary:
        summary = summary.split("Summary:")[-1].strip()
    
    return summary

## Run the Models and Generate Summaries

Let's run our three approaches on the test dataset and collect the results.

In [35]:
def generate_summaries(test_data, num_samples=5):
    """Generate summaries using different approaches."""
    results = []
    
    # Select a subset of examples for testing
    if len(test_data) > num_samples:
        test_subset = test_data[:num_samples]
    else:
        test_subset = test_data
    
    # For few-shot, select examples not in the test subset
    few_shot_examples = test_data[num_samples:num_samples+2] if len(test_data) > num_samples+2 else test_data[:2]
    
    for item in tqdm(test_subset, desc="Generating summaries"):
        result = {
            "id": item["id"],
            "text": item["text"],
            "reference_summary": item["summary"],
            "summaries": {}
        }
        
        # Zero-shot with base model
        result["summaries"]["zero_shot_base"] = zero_shot_summarize(
            base_model, base_tokenizer, item["text"]
        )
        
        # Few-shot with base model (using 2 examples)
        result["summaries"]["few_shot_base"] = few_shot_summarize(
            base_model, base_tokenizer, item["text"], few_shot_examples
        )
        
        # Fine-tuned model (if available)
        if finetuned_model is not None:
            result["summaries"]["finetuned"] = zero_shot_summarize(
                finetuned_model, finetuned_tokenizer, item["text"]
            )
        
        results.append(result)
    
    return results

# Generate summaries
summary_results = generate_summaries(test_data)

# Display the first result
if summary_results:
    print("\nFirst article summary comparisons:")
    first_result = summary_results[0]
    print("\nReference summary:")
    print(first_result["reference_summary"])
    
    print("\nZero-shot base model:")
    print(first_result["summaries"]["zero_shot_base"])
    
    print("\nFew-shot base model:")
    print(first_result["summaries"]["few_shot_base"])
    
    if "finetuned" in first_result["summaries"]:
        print("\nFine-tuned model:")
        print(first_result["summaries"]["finetuned"])

Generating summaries:   0%|          | 0/5 [00:00<?, ?it/s]


First article summary comparisons:

Reference summary:
Ricky Mulvey: I'm glad I'm glad to have that experience.
One of the themes that struck me from this is that, you know, people like to think, I'm a free markets person, or I want government involved with things.
I look mid century treasury markets and a pretty conservative head of the Federal Reserve who use policy to make those markets orderly.
Chris Hughes: I'm still writing a dissertation to finish a PhD it's on Fed history.
I'm going to sell my shares in Meta, and I'm going to give you that money for your national investment bank because I believe in you to allocate this, Chris.

Zero-shot base model:
Ricky Mulvey: One of the themes that struck me from this is that, you know, people like to think, I'm a free markets person, or I want government involved with things.

Few-shot base model:
Shares of NASDAQ:AAPL opened at $211.45 on Friday. Apple comprises 5.4% of GPM Growth Investors Inc.’s portfolio, making the stock its 4th big

## Standard Metrics Evaluation

Let's first evaluate using standard metrics like ROUGE and BLEU.

In [36]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

# Initialize ROUGE scorer
rouge_scorer_instance = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_standard_metrics(results):
    """Calculate ROUGE and BLEU scores for each summarization approach."""
    metrics = {
        "zero_shot_base": {"rouge1": [], "rouge2": [], "rougeL": [], "bleu": []},
        "few_shot_base": {"rouge1": [], "rouge2": [], "rougeL": [], "bleu": []},
        "finetuned": {"rouge1": [], "rouge2": [], "rougeL": [], "bleu": []} if "finetuned" in results[0]["summaries"] else None
    }
    
    for result in results:
        reference = result["reference_summary"]
        reference_tokens = reference.split()
        
        for approach, summary in result["summaries"].items():
            # Calculate ROUGE scores
            rouge_scores = rouge_scorer_instance.score(reference, summary)
            metrics[approach]["rouge1"].append(rouge_scores["rouge1"].fmeasure)
            metrics[approach]["rouge2"].append(rouge_scores["rouge2"].fmeasure)
            metrics[approach]["rougeL"].append(rouge_scores["rougeL"].fmeasure)
            
            # Calculate BLEU score
            smoothing = SmoothingFunction().method1
            summary_tokens = summary.split()
            try:
                bleu_score = sentence_bleu([reference_tokens], summary_tokens, smoothing_function=smoothing)
                metrics[approach]["bleu"].append(bleu_score)
            except:
                metrics[approach]["bleu"].append(0)
    
    # Calculate average scores
    avg_metrics = {}
    for approach, scores in metrics.items():
        if scores is not None:
            avg_metrics[approach] = {
                metric: np.mean(values) for metric, values in scores.items()
            }
    
    return avg_metrics

# Calculate metrics
standard_metrics = calculate_standard_metrics(summary_results)

# Display metrics
print("Average ROUGE and BLEU Scores:")
for approach, metrics in standard_metrics.items():
    print(f"\n{approach.replace('_', ' ').title()}:")
    for metric, score in metrics.items():
        print(f"  {metric}: {score:.4f}")

Average ROUGE and BLEU Scores:

Zero Shot Base:
  rouge1: 0.1625
  rouge2: 0.0833
  rougeL: 0.1403
  bleu: 0.0171

Few Shot Base:
  rouge1: 0.1193
  rouge2: 0.0236
  rougeL: 0.0725
  bleu: 0.0048

Finetuned:
  rouge1: 0.1952
  rouge2: 0.1070
  rougeL: 0.1637
  bleu: 0.0371
